# Validator Agent Notebook

This notebook demonstrates the Validator Agent which validates quantum circuits and uses LLM to fix broken code.

## Features
- **Local Validation**: Runs code using the **Amazon Braket SDK** (BraketCompiler + LocalSimulator)—no external backend required
- **LLM-Powered Fixing**: When code fails, the LLM analyzes errors and suggests fixes

## Prerequisites
1. **Amazon Braket SDK** installed (`pip install amazon-braket-sdk`)
2. AWS/LLM configured in `config/config.json` for validator model (for LLM fixing)

---

# Part 1: Setup & Local Braket SDK Check

Verify the Amazon Braket SDK is available and that local compile + simulate works.

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.braket_rag_code_assistant.config import get_config, setup_logging
from src.agents.validator import ValidatorAgent
from src.tools.compiler import BraketCompiler
from src.tools.simulator import QuantumSimulator

# Setup logging
setup_logging()
print("✅ Imports completed successfully.")

2026-03-16 20:43:10 | INFO     | src.braket_rag_code_assistant.config.logging:setup_all:137 | Logging configuration completed


✅ Imports completed successfully.


## 1.1 Check Local Braket SDK

Verify the Amazon Braket SDK is installed and that we can compile and simulate a small circuit locally.

In [2]:
# Use local Braket compiler and simulator (no external backend)
compiler = BraketCompiler()
simulator = QuantumSimulator()

# Minimal Braket code: compile and run locally
sample_code = """
from braket.circuits import Circuit
circuit = Circuit().h(0).cnot(0, 1)
"""

result = compiler.compile(sample_code, execute=True)

if result.get("success") and result.get("circuit") is not None:
    print("✅ Amazon Braket SDK is available (local compile OK)")
    sim_result = simulator.simulate(result["circuit"], repetitions=100)
    if sim_result.get("success"):
        print("✅ Local simulation OK")
        print(f"   Sample counts: {sim_result.get('histogram', {})}")
    else:
        print("⚠️ Compile OK but simulation failed:", sim_result.get("error"))
else:
    print("❌ Braket SDK compile failed")
    for e in result.get("errors", []):
        print(f"   {e}")

2026-03-16 20:43:10 | INFO     | src.tools.simulator:__init__:80 | Initialized default simulator
2026-03-16 20:43:10 | INFO     | src.tools.simulator:simulate:125 | Simulation completed. Histogram: {'11': 54, '00': 46}


✅ Amazon Braket SDK is available (local compile OK)
✅ Local simulation OK
   Sample counts: {'11': 54, '00': 46}


## 1.2 Local validation path

The Validator Agent runs code **locally** via `BraketCompiler` and `QuantumSimulator` (Amazon Braket SDK). No QCanvas or other backend is required.

In [3]:
# test_code used later in Part 2 for validation examples
test_code = """
from braket.circuits import Circuit
circuit = Circuit().h(0).cnot(0, 1)
print(circuit)
"""
print("✅ Part 1 complete. Validation will use the Amazon Braket SDK (local compiler + simulator).")

✅ Part 1 complete. Validation will use the Amazon Braket SDK (local compiler + simulator).


## 1.3 Part 1 summary

No external backend is needed. The Validator Agent uses **BraketCompiler** and **QuantumSimulator** (Amazon Braket SDK) to compile and run circuits locally.

In [4]:
# No QCanvas; simulation is done by ValidatorAgent via QuantumSimulator (Braket SDK)
print("Simulation will be run by ValidatorAgent using the local Braket SDK.")

Simulation will be run by ValidatorAgent using the local Braket SDK.


## 1.4 Part 1 complete

Ready to use the Validator Agent in **local** mode (Amazon Braket SDK).

In [5]:
# Part 1 done. ValidatorAgent will compile + simulate code locally (no QCanvas).
print("=" * 50)
print("Part 1 complete. Ready for Validator Agent (local Braket SDK).")
print("=" * 50)

Part 1 complete. Ready for Validator Agent (local Braket SDK).


---

# Part 2: Validator Agent Testing

The Validator Agent runs in **local** mode: it compiles and executes code using the Amazon Braket SDK (BraketCompiler + QuantumSimulator).

## 2.1 Initialize Validator Agent

In [6]:
# First, add imports and setup RAG components
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever

# Initialize Knowledge Base with all datasets
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
kb.load_from_directory()
kb.index_entries()
print(f"Loaded {len(kb.entries)} KB entries for semantic validation")

# Create retriever
retriever = Retriever(knowledge_base=kb)

# Initialize in LOCAL mode: uses Amazon Braket SDK (no QCanvas)
validator = ValidatorAgent(mode="local", retriever=retriever)

print("✅ ValidatorAgent initialized (local Braket SDK, RAG support)")
...
print(f"   Mode: {validator.mode}")
print(f"   LLM Enabled: {validator.llm_enabled}")
print(f"   Ollama Model: {validator.ollama_model}")
print(f"   Default Shots: {validator.default_shots}")


2026-03-16 20:43:10 | INFO     | config.config_loader:load:101 | ✅ Loaded configuration from D:\University\Uni\Hackathon\Amazons\config\config.json


2026-03-16 20:43:10,075 - botocore.credentials - INFO - credentials.py:1252 - Found credentials in environment variables.


2026-03-16 20:43:10 | INFO     | src.rag.embeddings:_init_aws:120 | Using AWS Nova Multimodal Embeddings: amazon.nova-2-multimodal-embeddings-v1:0, dim=1024
2026-03-16 20:43:10 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2026-03-16 20:43:10 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2026-03-16 20:43:10 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 1024
2026-03-16 20:43:10 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2026-03-16 20:43:10 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Hackathon\Amazons\data\knowledge_base\curated_designer_examples.jsonl
2026-03-16 20:43:10 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from D:\University\Uni\Hackathon\Amazons\data\knowledge_base\curated_designer_examples.jsonl
2026-03-16 20:43:10 | INFO     | src.rag.kno

Loaded 140 KB entries for semantic validation
✅ ValidatorAgent initialized (local Braket SDK, RAG support)
   Mode: local
   LLM Enabled: True
   Ollama Model: us.amazon.nova-premier-v1:0
   Default Shots: 1024


## 2.2 Validate Valid Code (Bell State)

In [7]:
valid_code = """
from braket.circuits import Circuit

# Create a Bell state circuit
q0, q1 = braket.LineQubit.range(2)
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1),
    braket.measure(q0, q1, key='result')
)
print(circuit)
"""

task = {
    "code": valid_code,
    "description": "Create a Bell state with two entangled qubits",
    "shots": 1024
}

print("🔍 Validating Bell state code...")
result = validator.execute(task)

print(f"\n{'✅' if result.get('success') else '❌'} Validation {'PASSED' if result.get('success') else 'FAILED'}")
print(f"Stage: {result.get('stage')}")

if result.get('success'):
    results = result.get('results', {})
    print(f"\n📊 Results:")
    print(f"   Counts: {results.get('counts', {})}")
    print(f"   Probabilities: {results.get('probs', {})}")
else:
    print(f"\n🔴 Error: {result.get('error')}")

2026-03-16 20:55:49 | INFO     | src.agents.validator:execute:133 | Validating code (attempt 1/3)...


🔍 Validating Bell state code...


2026-03-16 20:55:49 | INFO     | src.agents.validator:_execute_local:439 | Running LLM analysis for compilation error fixing...
2026-03-16 20:55:53 | INFO     | src.agents.validator:execute:168 | Using LLM-fixed code for retry...
2026-03-16 20:55:53 | INFO     | src.agents.validator:execute:135 | Re-validating fixed code (attempt 2/3)...
2026-03-16 20:55:53 | INFO     | src.agents.validator:_execute_local:439 | Running LLM analysis for compilation error fixing...
2026-03-16 20:55:56 | INFO     | src.agents.validator:execute:168 | Using LLM-fixed code for retry...
2026-03-16 20:55:56 | INFO     | src.agents.validator:execute:135 | Re-validating fixed code (attempt 3/3)...
2026-03-16 20:55:56 | INFO     | src.tools.simulator:simulate:125 | Simulation completed. Histogram: {'11': 517, '00': 507}
2026-03-16 20:55:56 | INFO     | src.agents.validator:execute:161 | Validation passed on attempt 3



✅ Validation PASSED
Stage: simulation

📊 Results:
   Counts: {'11': 517, '00': 507}
   Probabilities: None


## 2.3 Validate Invalid Code (with LLM Fix)

In [8]:
# Code with missing import and missing measurement
invalid_code = """
# Missing from braket.circuits import Circuit!
q0, q1 = braket.LineQubit.range(2)
circuit = braket.Circuit(
    braket.H(q0),
    braket.CNOT(q0, q1)
)
"""

task = {
    "code": invalid_code,
    "description": "Create a simple quantum circuit with H and CNOT gates",
    "force_llm_fix": True
}

print("🔍 Validating broken code (missing import)...")
result = validator.execute(task)

print(f"\n{'✅' if result.get('success') else '❌'} Validation {'PASSED' if result.get('success') else 'FAILED'}")
print(f"Stage: {result.get('stage')}")

if result.get('error'):
    print(f"\n🔴 Error: {result.get('error')}")

# Show LLM analysis
llm_analysis = result.get('llm_analysis', {})
if llm_analysis:
    print("\n🤖 LLM Analysis:")
    if llm_analysis.get('fixed_code'):
        print("\n📝 Fixed Code:")
        print("```python")
        print(llm_analysis['fixed_code'])
        print("```")
    if llm_analysis.get('analysis'):
        print(f"\n💡 Explanation: {llm_analysis['analysis']}")

2026-03-16 20:55:56 | INFO     | src.agents.validator:execute:133 | Validating code (attempt 1/3)...
2026-03-16 20:55:56 | INFO     | src.agents.validator:_execute_local:439 | Running LLM analysis for compilation error fixing...


🔍 Validating broken code (missing import)...


2026-03-16 20:55:59 | INFO     | src.agents.validator:execute:168 | Using LLM-fixed code for retry...
2026-03-16 20:55:59 | INFO     | src.agents.validator:execute:135 | Re-validating fixed code (attempt 2/3)...
2026-03-16 20:55:59 | INFO     | src.agents.validator:_execute_local:499 | Running LLM analysis for missing measurement fixing...
2026-03-16 20:56:01 | INFO     | src.agents.validator:execute:168 | Using LLM-fixed code for retry...
2026-03-16 20:56:01 | INFO     | src.agents.validator:execute:135 | Re-validating fixed code (attempt 3/3)...
2026-03-16 20:56:01 | INFO     | src.tools.simulator:simulate:125 | Simulation completed. Histogram: {'11': 512, '00': 512}
2026-03-16 20:56:01 | INFO     | src.agents.validator:_execute_local:565 | Running LLM analysis for code fixing...
2026-03-16 20:56:06 | INFO     | src.agents.validator:execute:161 | Validation passed on attempt 3



✅ Validation PASSED
Stage: simulation

🤖 LLM Analysis:

📝 Fixed Code:
```python
from braket.circuits import Circuit
from braket.devices import LocalSimulator

circuit = Circuit().h(0).cnot(0, 1)
device = LocalSimulator("default")
result = device.run(circuit, shots=1000).result()
counts = result.measurement_counts
print(f"Counts: {counts}")
```

💡 Explanation: The original code correctly created a Bell state circuit but didn't execute it through a device to get measurement probabilities. The simulation results showed raw counts but no probabilities because probabilities require explicit calculation. The fix adds:
1. Device initialization with LocalSimulator
2. Execution with 1000 shots to get statistical counts
3. Proper extraction of measurement_counts from the result
4. Removal of redundant `measure()` call since Braket automatically measures all qubits when unspecified


## 2.4 Re-validate Fixed Code

In [9]:
fixed_code = result.get('fixed_code') or result.get('llm_analysis', {}).get('fixed_code')

if fixed_code:
    print("🔍 Re-validating LLM-fixed code...")
    
    task_fixed = {
        "code": fixed_code,
        "description": "Fixed version of the circuit",
        "shots": 1024
    }
    
    result_fixed = validator.execute(task_fixed)
    
    print(f"\n{'✅' if result_fixed.get('success') else '❌'} Fixed code {'PASSED' if result_fixed.get('success') else 'FAILED'}")
    
    if result_fixed.get('success'):
        results = result_fixed.get('results', {})
        print(f"   Counts: {results.get('counts', {})}")
    else:
        print(f"   Error: {result_fixed.get('error')}")
else:
    print("⚠️ No fixed code available from previous step.")

2026-03-16 20:56:06 | INFO     | src.agents.validator:execute:133 | Validating code (attempt 1/3)...
2026-03-16 20:56:06 | INFO     | src.agents.validator:_execute_local:499 | Running LLM analysis for missing measurement fixing...


🔍 Re-validating LLM-fixed code...


2026-03-16 20:56:08 | INFO     | src.agents.validator:execute:168 | Using LLM-fixed code for retry...
2026-03-16 20:56:08 | INFO     | src.agents.validator:execute:135 | Re-validating fixed code (attempt 2/3)...
2026-03-16 20:56:08 | INFO     | src.tools.simulator:simulate:125 | Simulation completed. Histogram: {'00': 516, '11': 508}
2026-03-16 20:56:09 | INFO     | src.agents.validator:execute:161 | Validation passed on attempt 2



✅ Fixed code PASSED
   Counts: {'00': 516, '11': 508}


---
# Part 3: Comprehensive Test Suite

Run the full suite of 20+ test cases to verify the Validator's robustness.

In [10]:
from tests.test_validator_circuits import run_tests

# Run all test cases
df_results = run_tests(validator)

# Display summary
print("\n" + "="*40)
print("TEST SUITE SUMMARY")
print("="*40)
print(f"Total Tests: {len(df_results)}")
print(f"Passed: {len(df_results[df_results['Status'].str.contains('PASS')])}")
print(f"Failed: {len(df_results[df_results['Status'].str.contains('FAIL|ERROR')])}")

# Show full table
df_results

📝 Logging execution details to: D:\University\Uni\Hackathon\Amazons\tests\test_execution.log
🚀 Starting Validator Test Suite (22 cases)...

[1/22] Testing: Teleportation - Valid... 

20:56:09 | Validating code (attempt 1/3)...
20:56:09 | Simulation completed. Histogram: {'111': 114, '011': 122, '000': 129, '101': 149, '001': 116, '100': 121, '110': 134, '010': 139}
20:56:09 | Running LLM analysis for code fixing...
20:56:12 | Bedrock throttled, waiting %ds before retry (%d/%d)
20:56:31 | Validation passed on attempt 1


-> PASS (22.65s)
[2/22] Testing: Teleportation - Missing Import... 

20:56:31 | Validating code (attempt 1/3)...
20:56:31 | Running LLM analysis for compilation error fixing...
20:56:33 | Using LLM-fixed code for retry...
20:56:33 | Re-validating fixed code (attempt 2/3)...
20:56:33 | Running LLM analysis for missing measurement fixing...
20:56:36 | Bedrock throttled, waiting %ds before retry (%d/%d)
20:56:53 | Using LLM-fixed code for retry...
20:56:53 | Re-validating fixed code (attempt 3/3)...
20:56:53 | Simulation completed. Histogram: {'0': 526, '1': 498}
20:56:53 | Running LLM analysis for code fixing...
20:56:58 | Validation passed on attempt 3


-> PASS (Fixed) (26.22s)
[3/22] Testing: Teleportation - Typo in Module... 

...

21:05:20 | Validating code (attempt 1/3)...
21:05:20 | No LLM fix available, retrying with same code...
21:05:20 | Re-validating fixed code (attempt 2/3)...
21:05:20 | No LLM fix available, retrying with same code...
21:05:20 | Re-validating fixed code (attempt 3/3)...
21:05:20 | Validation failed after 3 attempts


-> PASS (Detected) (0.00s)
[22/22] Testing: Misc - Natural Language... 

21:05:20 | Validating code (attempt 1/3)...
21:05:20 | Running LLM analysis for compilation error fixing...
21:05:24 | Bedrock throttled, waiting %ds before retry (%d/%d)
21:05:41 | Using LLM-fixed code for retry...
21:05:41 | Re-validating fixed code (attempt 2/3)...
21:05:41 | Simulation completed. Histogram: {'0': 495, '1': 529}
21:05:41 | Running LLM analysis for code fixing...
21:05:47 | Validation passed on attempt 2


-> PASS (Fixed) (26.88s)

TEST SUITE SUMMARY
Total Tests: 22
Passed: 22
Failed: 0


,ID,Name,Description,Expected,Actual_Success,Has_Fix,Validation_Attempts,Status,Error_Msg
0,1,Teleportation - Valid,Standard Teleportation circuit using Braket SDK,PASS,True,True,1,PASS,
1,2,Teleportation - Missing Import,Using Circuit without importing it,FAIL,True,True,3,PASS (Fixed),
2,3,Teleportation - Typo in Module,Typo 'Circuite' instead of 'Circuit',FAIL,True,True,3,PASS (Fixed),
3,4,Teleportation - Undefined Variable,Using undefined variable q3,FAIL,False,True,3,PASS (Fixed),'LocalQuantumTask' object has no attribute 're...
4,5,Teleportation - Syntax Error,Missing closing parenthesis,FAIL,True,True,3,PASS (Fixed),
5,6,Deutsch-Jozsa - Valid,Standard DJ circuit for 2 qubits using Braket,PASS,True,True,1,PASS,
6,7,Deutsch-Jozsa - Indentation Error,Python indentation error in loop,FAIL,True,True,3,PASS (Fixed),
7,8,Deutsch-Jozsa - Wrong Gate Usage,Passing extra argument to H gate,FAIL,False,True,3,PASS (Fixed),'LocalQuantumTask' object has no attribute 're...
8,9,Deutsch-Jozsa - Logic/Type Error,Passing a string instead of qubit index,FAIL,False,True,3,PASS (Fixed),'LocalQuantumTask' object has no attribute 're...
9,10,Deutsch-Jozsa - Missing Measurement,Circuit has no measurements,FAIL,True,True,2,PASS (Fixed),
